# Hardware-Aware RL Model Evaluator

Evaluate the logged PPO policy trained with two-qubit weight 1.0, depth weight 0.0, runtime weight 0.0, and final-reward weight 1.0. Compare it with all three single optimizers and all nine ordered two-step chains on the held-out test split.


In [ ]:
import os
import textwrap
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.mplconfig'))

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sb3_contrib import MaskablePPO

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'dbs' / 'hardware_aware_combined' / 'combined_master.csv').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not locate the hardware-aware combined master.')
    PROJECT_ROOT = PROJECT_ROOT.parent

PACKAGE_DIR = PROJECT_ROOT / 'machine_learning' / 'RL' / 'hardware_aware_two_opt_chain_rl'
if str(PACKAGE_DIR) not in __import__('sys').path:
    __import__('sys').path.insert(0, str(PACKAGE_DIR))

from data_processing import default_master_csv_path, split_summary_frame
from train import build_env, reward_splits
from evaluation import evaluate_model_on_frame
from rewards import single_step_reward, incremental_step_reward, compute_final_correction

TRAINING_LOG_PATH = PACKAGE_DIR / 'training_runs.csv'
OUTPUT_DIR = PACKAGE_DIR / 'model_evaluator_outputs'
SAVE_OUTPUTS = True
TARGET_MODEL_PATH = PACKAGE_DIR / 'models' / 'hardware_aware_optimizer_chain_ppo_current_reward_seed42_fbplus_all_features_rw0p0_tqw1p0_dw0p0_frw1p0.zip'
EVAL_SPLIT = 'test'

OPTIMIZER_LABELS = {
    'qiskit_ai': 'Qiskit AI',
    'qiskit_standard': 'Qiskit standard',
    'tket': 'TKET',
}

sns.set_theme(
    style='whitegrid',
    context='notebook',
    rc={
        'figure.facecolor': '#FFFFFF',
        'axes.facecolor': '#FBFAF7',
        'axes.edgecolor': '#AAB3B9',
        'axes.labelcolor': '#26343D',
        'text.color': '#26343D',
        'xtick.color': '#34434C',
        'ytick.color': '#34434C',
        'grid.color': '#D8DEE2',
        'grid.linewidth': 0.7,
        'axes.titleweight': 'bold',
    },
)

if EVAL_SPLIT not in {'train', 'validation', 'test'}:
    raise ValueError('EVAL_SPLIT must be train, validation, or test.')
if not TRAINING_LOG_PATH.exists():
    raise FileNotFoundError(f'Training log not found: {TRAINING_LOG_PATH}')


## Select the exact retrained model

The evaluator is pinned to `hardware_aware_optimizer_chain_ppo_current_reward_seed42_fbplus_all_features_rw0p0_tqw1p0_dw0p0_frw1p0.zip`. It reads the matching run metadata from `training_runs.csv` and raises an error if that exact model or log entry is missing. The model path is shown in the run summary below.


In [ ]:
training_runs = pd.read_csv(TRAINING_LOG_PATH)
if training_runs.empty:
    raise ValueError(f'No model runs are recorded in {TRAINING_LOG_PATH}.')
if not TARGET_MODEL_PATH.exists():
    raise FileNotFoundError(f'Target model not found: {TARGET_MODEL_PATH}')

logged_paths = training_runs['model_path'].map(
    lambda value: Path(str(value)).expanduser().resolve()
)
matching_runs = training_runs.loc[logged_paths == TARGET_MODEL_PATH.resolve()]
if matching_runs.empty:
    raise ValueError(f'Target model is not recorded in {TRAINING_LOG_PATH}: {TARGET_MODEL_PATH}')
model_run = matching_runs.iloc[-1]
MODEL_PATH = TARGET_MODEL_PATH.resolve()

FEATURE_BATCH = str(model_run['feature_batch'])
REWARD_TYPE = str(model_run['reward_type'])
RUNTIME_WEIGHT = float(model_run['runtime_weight'])
TWO_QUBIT_WEIGHT = float(model_run['two_qubit_weight'])
DEPTH_WEIGHT = float(model_run['depth_weight'])
FINAL_REWARD_WEIGHT = float(model_run['final_reward_weight'])
MODEL_SEED = int(model_run['algorithm_seed'])
if (FEATURE_BATCH, REWARD_TYPE, RUNTIME_WEIGHT, TWO_QUBIT_WEIGHT, DEPTH_WEIGHT, FINAL_REWARD_WEIGHT) != (
    'plus_all_features', 'current_reward', 0.0, 1.0, 0.0, 1.0
):
    raise ValueError(f'Target model metadata does not match the requested settings: {model_run.to_dict()}')
MODEL_LABEL = 'RL policy · 2Q=1 · depth=0 · runtime=0 · final=1'

selected_run_summary = pd.DataFrame([{
    'model': MODEL_LABEL,
    'path': str(MODEL_PATH),
    'evaluation_split': EVAL_SPLIT,
    'feature_batch': FEATURE_BATCH,
    'reward_type': REWARD_TYPE,
    'runtime_weight': RUNTIME_WEIGHT,
    'two_qubit_weight': TWO_QUBIT_WEIGHT,
    'depth_weight': DEPTH_WEIGHT,
    'final_reward_weight': FINAL_REWARD_WEIGHT,
    'hyperparameter_search': model_run['used_hyperparameter_grid_search'],
    'hyperparameters': model_run['hyperparameters'],
}])
display(selected_run_summary)


## Prepare matching circuit splits

The evaluator uses the same master, random split seed, reward definition, and feature batch as the selected model. The model is scored once on each circuit in the selected split.


In [ ]:
bundle = reward_splits(
    csv_path=default_master_csv_path(),
    runtime_weight=RUNTIME_WEIGHT,
    reward_type=REWARD_TYPE,
    two_qubit_weight=TWO_QUBIT_WEIGHT,
    depth_weight=DEPTH_WEIGHT,
    save_splits=False,
    feature_batch=FEATURE_BATCH,
)

optimizer_names = bundle['optimizer_names']
category_names = bundle['category_names']
metric_scale = bundle['metric_scale']
observation_feature_scale = bundle['observation_feature_scale']
static_feature_columns = bundle['static_feature_columns']
dynamic_feature_columns = bundle['dynamic_feature_columns']
include_category = bundle['feature_batch_config']['include_category']

split_frames = {
    'train': bundle['train_df'].copy(),
    'validation': bundle['validation_df'].copy(),
    'test': bundle['test_df'].copy(),
}

def add_episode_reward(frame):
    result = frame.copy()
    row_lookup = {
        (row.original_circuit, row.chain_name): row._asdict()
        for row in result.itertuples(index=False)
    }
    rewards = []
    for row in result.itertuples(index=False):
        row_data = row._asdict()
        if '__' not in row.chain_name:
            reward = single_step_reward(
                row_data,
                reward_type=REWARD_TYPE,
                runtime_weight=RUNTIME_WEIGHT,
                two_qubit_weight=TWO_QUBIT_WEIGHT,
                depth_weight=DEPTH_WEIGHT,
            )
        else:
            first_optimizer = row.chain_name.split('__', 1)[0]
            first_row = row_lookup[(row.original_circuit, first_optimizer)]
            reward = single_step_reward(
                first_row,
                reward_type=REWARD_TYPE,
                runtime_weight=RUNTIME_WEIGHT,
                two_qubit_weight=TWO_QUBIT_WEIGHT,
                depth_weight=DEPTH_WEIGHT,
            )
            reward += incremental_step_reward(
                row_data,
                first_row,
                reward_type=REWARD_TYPE,
                runtime_weight=RUNTIME_WEIGHT,
                two_qubit_weight=TWO_QUBIT_WEIGHT,
                depth_weight=DEPTH_WEIGHT,
            )
        reward += compute_final_correction(
            row_data,
            reward_type=REWARD_TYPE,
            final_reward_weight=FINAL_REWARD_WEIGHT,
            two_qubit_weight=TWO_QUBIT_WEIGHT,
            depth_weight=DEPTH_WEIGHT,
        )
        rewards.append(float(reward))

    result['episode_reward_if_selected'] = rewards
    result['episode_total_runtime_seconds'] = pd.to_numeric(
        result['total_runtime_seconds'], errors='coerce'
    )
    result['episode_runtime_score'] = pd.to_numeric(
        result['total_runtime_score'], errors='coerce'
    )
    return result

split_frames = {name: add_episode_reward(frame) for name, frame in split_frames.items()}
eval_df = split_frames[EVAL_SPLIT].copy()
display(split_summary_frame(split_frames))
print(f'Evaluating {eval_df["original_circuit"].nunique():,} circuits from the {EVAL_SPLIT} split.')


## Run the policy

The environment masks stop on the first decision, so every policy episode selects a first-round optimizer. The policy can stop after that result or choose a second optimizer.


In [ ]:
model = MaskablePPO.load(str(MODEL_PATH))
rl_selected = evaluate_model_on_frame(
    model,
    eval_df,
    build_env,
    optimizer_names,
    category_names,
    metric_scale,
    observation_feature_scale,
    static_feature_columns,
    dynamic_feature_columns,
    include_category,
    method_name=MODEL_LABEL,
    reward_type=REWARD_TYPE,
    runtime_weight=RUNTIME_WEIGHT,
    two_qubit_weight=TWO_QUBIT_WEIGHT,
    depth_weight=DEPTH_WEIGHT,
    final_reward_weight=FINAL_REWARD_WEIGHT,
)

if len(rl_selected) != eval_df['original_circuit'].nunique():
    raise ValueError('The policy did not return one result for every evaluation circuit.')
if rl_selected['selected_chain'].isna().any():
    raise ValueError('The policy returned an empty chain selection.')
if rl_selected['selected_chain'].eq('__original__').any():
    raise ValueError('The policy selected no optimization despite the first-step mask.')

rl_selected['method'] = MODEL_LABEL
rl_selected['method_kind'] = 'RL policy'
print(f'Policy selections: {len(rl_selected):,}; first-round stop choices: 0')


## Build baseline strategies

The fixed-path baseline is selected using validation reward and evaluated on the test split. Per-circuit oracle baselines select the best observed result for each test circuit; they are upper bounds, not deployable policies. No optimization is shown as a reference but is unavailable to the RL policy.


In [ ]:
def selected_rows(frame, chain_name, method, kind):
    rows = frame[frame['chain_name'] == chain_name].copy()
    rows['method'] = method
    rows['method_kind'] = kind
    rows['selected_chain'] = chain_name
    rows['episode_reward'] = rows['episode_reward_if_selected']
    rows['episode_total_runtime_seconds'] = rows['total_runtime_seconds']
    rows['episode_runtime_score'] = rows['total_runtime_score']
    return rows


def no_optimization_rows(frame):
    rows = frame.groupby('original_circuit', sort=True).head(1).copy()
    rows['chain_name'] = '__original__'
    rows['selected_chain'] = '__original__'
    for metric in ['final_num_qubits', 'final_depth', 'final_two_qubit_gates', 'final_total_gates', 'final_two_qubit_depth']:
        rows[metric] = rows[metric.replace('final_', 'initial_')]
    zero_columns = [
        'total_runtime_seconds', 'total_runtime_score', 'total_two_qubit_gate_reduction',
        'total_two_qubit_gate_reduction_pct', 'total_depth_reduction', 'total_depth_reduction_pct',
        'two_qubit_gate_reduction', 'two_qubit_gate_reduction_pct', 'depth_reduction',
        'depth_reduction_pct', 'current_step_two_qubit_gate_reduction',
        'current_step_two_qubit_gate_reduction_pct', 'current_step_depth_reduction',
        'current_step_depth_reduction_pct', 'runtime_score', 'local_step_reward',
        'utility_reward', 'episode_reward_if_selected', 'episode_total_runtime_seconds',
        'episode_runtime_score',
    ]
    rows[zero_columns] = 0.0
    return rows


def oracle_rows(frame, value_column, *, maximize, method):
    order = frame.sort_values(
        ['original_circuit', value_column, 'total_runtime_seconds'],
        ascending=[True, not maximize, True],
    )
    rows = order.groupby('original_circuit', sort=True).head(1).copy()
    rows['method'] = method
    rows['method_kind'] = 'Per-circuit oracle'
    rows['selected_chain'] = rows['chain_name']
    rows['episode_reward'] = rows['episode_reward_if_selected']
    rows['episode_total_runtime_seconds'] = rows['total_runtime_seconds']
    rows['episode_runtime_score'] = rows['total_runtime_score']
    return rows


def summarize_method(frame, method, kind):
    return {
        'method': method,
        'method_kind': kind,
        'circuits': frame['original_circuit'].nunique(),
        'avg_reward_utility': frame['episode_reward'].mean(),
        'avg_pct_reduction': frame['total_two_qubit_gate_reduction_pct'].mean(),
        'avg_depth_reduction': frame['total_depth_reduction_pct'].mean(),
        'avg_runtime_seconds': frame['episode_total_runtime_seconds'].mean(),
        'avg_runtime_score': frame['episode_runtime_score'].mean(),
    }

train_df = split_frames['train']
single_chains = [name for name in optimizer_names if name in set(train_df['chain_name'])]
double_chains = [f'{first}__{second}' for first in optimizer_names for second in optimizer_names]

def best_chain(frame, candidates, column):
    scores = frame[frame['chain_name'].isin(candidates)].groupby('chain_name')[column].mean()
    if scores.empty:
        raise ValueError(f'No training outcomes found for {column}.')
    return scores.idxmax()

best_fixed_reward_chain = best_chain(train_df, single_chains + double_chains, 'episode_reward_if_selected')
best_single_optimizer_for_depth_reduction = best_chain(train_df, single_chains, 'total_depth_reduction_pct')
best_single_optimizer_for_two_qubit_reduction = best_chain(train_df, single_chains, 'total_two_qubit_gate_reduction_pct')
best_chain_for_depth_reduction = best_chain(train_df, double_chains, 'total_depth_reduction_pct')
best_chain_for_two_qubit_reduction = best_chain(train_df, double_chains, 'total_two_qubit_gate_reduction_pct')

best_reward_chain_label = 'Best fixed reward chain'
best_depth_chain_label = 'Best depth chain'
best_twoq_chain_label = 'Best 2Q chain'
best_depth_single_label = f'Best depth single: {OPTIMIZER_LABELS[best_single_optimizer_for_depth_reduction]}'
best_twoq_single_label = f'Best 2Q single: {OPTIMIZER_LABELS[best_single_optimizer_for_two_qubit_reduction]}'
best_possible_reward_label = 'Best possible reward (oracle)'
best_possible_depth_label = 'Best possible depth (oracle)'
best_possible_twoq_label = 'Best possible 2Q reduction (oracle)'

# Include no optimization when computing the per-circuit oracle references.
no_opt_eval = no_optimization_rows(eval_df)
oracle_candidates = pd.concat([eval_df, no_opt_eval], ignore_index=True)

def fixed_baseline(chain_name, label, kind):
    return selected_rows(eval_df, chain_name, label, kind)

baseline_selected_frames = {
    best_reward_chain_label: fixed_baseline(best_fixed_reward_chain, best_reward_chain_label, 'Fixed chain'),
    best_depth_chain_label: fixed_baseline(best_chain_for_depth_reduction, best_depth_chain_label, 'Fixed chain'),
    best_twoq_chain_label: fixed_baseline(best_chain_for_two_qubit_reduction, best_twoq_chain_label, 'Fixed chain'),
    best_depth_single_label: fixed_baseline(best_single_optimizer_for_depth_reduction, best_depth_single_label, 'Single optimizer'),
    best_twoq_single_label: fixed_baseline(best_single_optimizer_for_two_qubit_reduction, best_twoq_single_label, 'Single optimizer'),
    best_possible_reward_label: oracle_rows(oracle_candidates, 'episode_reward_if_selected', maximize=True, method=best_possible_reward_label),
    best_possible_depth_label: oracle_rows(oracle_candidates, 'total_depth_reduction_pct', maximize=True, method=best_possible_depth_label),
    best_possible_twoq_label: oracle_rows(oracle_candidates, 'total_two_qubit_gate_reduction_pct', maximize=True, method=best_possible_twoq_label),
}

# Compare each optimizer by itself, plus the best chain and reward oracle.
optimizer_selected_frames = {
    f'Single optimizer: {OPTIMIZER_LABELS[name]}': fixed_baseline(
        name, f'Single optimizer: {OPTIMIZER_LABELS[name]}', 'Single optimizer'
    )
    for name in single_chains
}

baseline_reference = pd.DataFrame([
    {'baseline': best_reward_chain_label, 'chain': best_fixed_reward_chain},
    {'baseline': best_depth_chain_label, 'chain': best_chain_for_depth_reduction},
    {'baseline': best_twoq_chain_label, 'chain': best_chain_for_two_qubit_reduction},
    {'baseline': best_depth_single_label, 'chain': best_single_optimizer_for_depth_reduction},
    {'baseline': best_twoq_single_label, 'chain': best_single_optimizer_for_two_qubit_reduction},
    {'baseline': best_possible_reward_label, 'chain': 'Per-circuit oracle, including no optimization'},
    {'baseline': best_possible_depth_label, 'chain': 'Per-circuit oracle, including no optimization'},
    {'baseline': best_possible_twoq_label, 'chain': 'Per-circuit oracle, including no optimization'},
])
display(baseline_reference)

comparison_frames = {
    **baseline_selected_frames,
    'No optimization (reference)': no_opt_eval.assign(
        method='No optimization (reference)', method_kind='Reference only',
        episode_reward=no_opt_eval['episode_reward_if_selected'],
        episode_total_runtime_seconds=no_opt_eval['total_runtime_seconds'],
        episode_runtime_score=no_opt_eval['total_runtime_score'],
    ),
}
comparison = pd.DataFrame([
    summarize_method(rl_selected, MODEL_LABEL, 'RL policy'),
    *[
        summarize_method(frame, label, frame['method_kind'].iloc[0])
        for label, frame in comparison_frames.items()
    ],
]).sort_values('avg_reward_utility', ascending=False).reset_index(drop=True)
display(comparison)


## Compare overall performance

Positive gate and depth reduction means fewer gates or lower depth than the original circuit. Negative values indicate a regression, which can occur when hardware mapping adds routing overhead. Runtime is total optimizer time.


In [ ]:
# Shared figure styling and file output, adapted from the v4 model evaluator.
PAPER_FIGURE_COLOR = '#FFFFFF'
PAPER_AXES_COLOR = '#F7F4EA'
PAPER_GRID_COLOR = '#CDD5D9'
PAPER_PANEL_EDGE = '#657681'
PAPER_BAR_EDGE = '#40515F'
PAPER_LABEL_COLOR = '#24333D'
PAPER_TRACK_COLOR = '#DFE5E7'
PAPER_BAR_HEIGHT = 0.58
PAPER_FIG_WIDTH = 9.0
PAPER_FIG_HEIGHT = 6.0

def prettify_label(name, wrap_width=24):
    name = str(name).replace('qiskit_ai', 'Qiskit AI')
    name = name.replace('qiskit_standard', 'Qiskit standard').replace('tket', 'TKET')
    name = name.replace('__', ' → ').replace('_', ' ')
    return textwrap.fill(name, width=wrap_width) if wrap_width else name

def style_publication_axes(ax, grid_axis=None):
    ax.figure.set_facecolor(PAPER_FIGURE_COLOR)
    ax.set_facecolor(PAPER_AXES_COLOR)
    ax.set_axisbelow(True)
    ax.tick_params(axis='both', colors=PAPER_LABEL_COLOR, width=0.8)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(PAPER_PANEL_EDGE)
        spine.set_linewidth(0.8)
    ax.grid(False)
    if grid_axis == 'x':
        ax.xaxis.grid(True, color=PAPER_GRID_COLOR, linewidth=0.65, alpha=0.8)
    elif grid_axis == 'y':
        ax.yaxis.grid(True, color=PAPER_GRID_COLOR, linewidth=0.65, alpha=0.8)

def save_figure(fig, filename):
    if SAVE_OUTPUTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / filename, bbox_inches='tight', dpi=300, facecolor=fig.get_facecolor())

def horizontal_bar(data, column, xlabel, title, filename, color='#6F83A6', fmt='{:.3f}', wrap_width=27):
    plot_data = data.sort_values(column, ascending=False).copy()
    labels = [prettify_label(value, wrap_width) for value in plot_data['method']]
    values = plot_data[column].astype(float).to_numpy()
    extent = max(float(np.nanmax(np.abs(values))), 1.0e-9)
    height = max(PAPER_FIG_HEIGHT, 0.43 * len(plot_data) + 1.8)
    fig, ax = plt.subplots(figsize=(PAPER_FIG_WIDTH, height), facecolor=PAPER_FIGURE_COLOR)
    style_publication_axes(ax, grid_axis='x')
    ypos = np.arange(len(plot_data))
    if np.nanmin(values) < 0:
        ax.barh(ypos, 2 * extent, left=-extent, color=PAPER_TRACK_COLOR, edgecolor=PAPER_BAR_EDGE, linewidth=0.6, height=PAPER_BAR_HEIGHT)
        bars = ax.barh(ypos, values, color=color, edgecolor=PAPER_BAR_EDGE, linewidth=0.8, height=PAPER_BAR_HEIGHT)
        ax.axvline(0, color=PAPER_PANEL_EDGE, linewidth=0.9)
        ax.set_xlim(-extent * 1.1, extent * 1.16)
    else:
        ax.barh(ypos, extent, color=PAPER_TRACK_COLOR, edgecolor=PAPER_BAR_EDGE, linewidth=0.6, height=PAPER_BAR_HEIGHT)
        bars = ax.barh(ypos, values, color=color, edgecolor=PAPER_BAR_EDGE, linewidth=0.8, height=PAPER_BAR_HEIGHT)
        ax.set_xlim(0, extent * 1.16)
    ax.set_yticks(ypos, labels)
    ax.invert_yaxis()
    ax.set_title(title, fontsize=17, pad=10, weight='semibold', color=PAPER_LABEL_COLOR)
    ax.set_xlabel(xlabel, fontsize=12, labelpad=8)
    ax.set_ylabel('')
    ax.tick_params(axis='x', labelsize=10)
    ax.tick_params(axis='y', labelsize=10, length=3, pad=6)
    pad = 0.012 * extent
    for bar, value in zip(bars, values):
        y = bar.get_y() + bar.get_height() / 2
        if value > 0 and value > 0.18 * extent:
            x, ha = value - pad, 'right'
        elif value < 0 and abs(value) > 0.18 * extent:
            x, ha = value + pad, 'left'
        elif value < 0:
            x, ha = value - pad, 'right'
        else:
            x, ha = value + pad, 'left'
        ax.text(x, y, fmt.format(value), va='center', ha=ha, fontsize=9, color=PAPER_LABEL_COLOR)
    fig.subplots_adjust(left=0.36, right=0.97, bottom=0.12, top=0.90)
    save_figure(fig, filename)
    plt.show()
    plt.close(fig)

# Selection heatmap: policy, selected fixed baselines, and each oracle's choices.
plot_frames = {MODEL_LABEL: rl_selected, **baseline_selected_frames}
selected_chain_counts = pd.concat(
    [frame[['method', 'selected_chain']] for frame in plot_frames.values()], ignore_index=True
).groupby(['method', 'selected_chain']).size().reset_index(name='circuits')
selected_chain_grid = selected_chain_counts.pivot_table(
    index='method', columns='selected_chain', values='circuits', fill_value=0, aggfunc='sum'
)
selected_chain_grid = selected_chain_grid.loc[
    :, selected_chain_grid.sum(axis=0).sort_values(ascending=False).index
]
selected_chain_grid.index = [prettify_label(name, 28) for name in selected_chain_grid.index]
selected_chain_grid.columns = [prettify_label(name, 17) for name in selected_chain_grid.columns]
fig, ax = plt.subplots(figsize=(max(10, 0.58 * len(selected_chain_grid.columns) + 3), max(6, 0.52 * len(selected_chain_grid.index) + 2.5)), facecolor=PAPER_FIGURE_COLOR)
style_publication_axes(ax)
sns.heatmap(selected_chain_grid, annot=True, fmt='d', cmap=sns.light_palette('#355F78', as_cmap=True), linewidths=0.5, linecolor=PAPER_AXES_COLOR, cbar_kws={'label': f'Number of {EVAL_SPLIT} circuits', 'shrink': 0.9}, ax=ax)
ax.set_title('Optimizer Chain Selection Frequency by Method', fontsize=16, pad=10, weight='semibold')
ax.set_xlabel('Selected optimizer chain', labelpad=9)
ax.set_ylabel('Method', labelpad=9)
plt.setp(ax.get_xticklabels(), rotation=55, ha='right', rotation_mode='anchor')
plt.setp(ax.get_yticklabels(), rotation=0)
fig.tight_layout()
save_figure(fig, 'selected_chain_counts_heatmap.pdf')
plt.show()
plt.close(fig)

# Raw path usage for the learned policy.
model_chain_counts = rl_selected['selected_chain'].value_counts().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(max(10, 0.9 * len(model_chain_counts) + 4), 6), facecolor=PAPER_FIGURE_COLOR)
style_publication_axes(ax, grid_axis='y')
labels = [prettify_label(name, None) for name in model_chain_counts.index]
bars = ax.bar(labels, model_chain_counts.values, color='#6F83A6', edgecolor=PAPER_BAR_EDGE, linewidth=0.8)
ax.set_title('RL Policy Chain Usage', fontsize=17, pad=12, weight='semibold')
ax.set_xlabel('Selected optimizer chain', fontsize=12, labelpad=8)
ax.set_ylabel('Number of selections', fontsize=12, labelpad=8)
ax.tick_params(axis='x', labelrotation=55, labelsize=10)
ax.tick_params(axis='y', labelsize=10)
for bar, value in zip(bars, model_chain_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, value, f'{value}', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
save_figure(fig, 'model_chain_usage.pdf')
plt.show()
plt.close(fig)

# Overall baseline comparisons, with oracle rows retained as labeled upper bounds.
utility_plot = comparison.copy()
twoq_plot = comparison[comparison['method'] != best_possible_depth_label].copy()
depth_plot = comparison[comparison['method'] != best_possible_twoq_label].copy()
runtime_plot = comparison[~comparison['method'].isin([best_possible_depth_label, best_possible_twoq_label])].copy()
horizontal_bar(utility_plot, 'avg_reward_utility', 'Average utility reward', 'Average Reward Utility by Method', 'avg_reward_utility.pdf')
horizontal_bar(twoq_plot, 'avg_pct_reduction', 'Average two-qubit gate reduction (%)', 'Average Two-Qubit Gate Reduction by Method', 'avg_pct_reduction.pdf', color='#4F7F8C', fmt='{:.1f}%')
horizontal_bar(depth_plot, 'avg_depth_reduction', 'Average depth reduction (%)', 'Average Depth Reduction by Method', 'avg_depth_reduction.pdf', color='#4F7F8C', fmt='{:.1f}%')
horizontal_bar(runtime_plot, 'avg_runtime_seconds', 'Average runtime (s)', 'Average Runtime by Method', 'avg_runtime_seconds.pdf', color='#C47A3A', fmt='{:.2f}')
horizontal_bar(runtime_plot, 'avg_runtime_score', 'Average runtime score', 'Average Runtime Score by Method', 'avg_runtime_score.pdf', color='#B05A62', fmt='{:.3f}')

# Optimizer-focused table, four standalone plots, and a 2x2 performance panel.
optimizer_comparison_rows = [summarize_method(frame, label, 'Single optimizer') for label, frame in optimizer_selected_frames.items()]
optimizer_comparison_rows.extend([
    summarize_method(baseline_selected_frames[best_twoq_chain_label], best_twoq_chain_label, 'Fixed chain'),
    summarize_method(baseline_selected_frames[best_possible_reward_label], best_possible_reward_label, 'Per-circuit oracle'),
    summarize_method(rl_selected, MODEL_LABEL, 'RL policy'),
])
optimizer_comparison = pd.DataFrame(optimizer_comparison_rows).sort_values('avg_reward_utility', ascending=False).reset_index(drop=True)
display(optimizer_comparison)
for column, xlabel, title, color, fmt, filename in [
    ('avg_reward_utility', 'Average utility reward', 'RL Policy vs Optimizers: Reward Utility', '#6F83A6', '{:.3f}', 'optimizer_only_avg_reward_utility.pdf'),
    ('avg_pct_reduction', 'Average two-qubit gate reduction (%)', 'RL Policy vs Optimizers: Two-Qubit Gate Reduction', '#4F8A70', '{:.1f}%', 'optimizer_only_avg_pct_reduction.pdf'),
    ('avg_depth_reduction', 'Average depth reduction (%)', 'RL Policy vs Optimizers: Depth Reduction', '#4F7F8C', '{:.1f}%', 'optimizer_only_avg_depth_reduction.pdf'),
    ('avg_runtime_seconds', 'Average runtime (s)', 'RL Policy vs Optimizers: Runtime', '#C47A3A', '{:.2f}', 'optimizer_only_avg_runtime_seconds.pdf'),
]:
    horizontal_bar(optimizer_comparison, column, xlabel, title, filename, color=color, fmt=fmt, wrap_width=19)

panel_data = pd.concat([
    optimizer_comparison,
    pd.DataFrame([summarize_method(baseline_selected_frames[best_depth_chain_label], best_depth_chain_label, 'Fixed chain')]),
], ignore_index=True)
fig, axes = plt.subplots(2, 2, figsize=(18, 9), facecolor=PAPER_FIGURE_COLOR)
for ax, (column, xlabel, color, fmt) in zip(axes.flat, [
    ('avg_reward_utility', 'Average utility reward', '#6F83A6', '{:.3f}'),
    ('avg_pct_reduction', 'Average 2Q gate reduction (%)', '#4F8A70', '{:.1f}%'),
    ('avg_depth_reduction', 'Average depth reduction (%)', '#4F7F8C', '{:.1f}%'),
    ('avg_runtime_seconds', 'Average runtime (s)', '#C47A3A', '{:.2f}'),
]):
    data = panel_data.sort_values(column, ascending=False).copy()
    values = data[column].astype(float).to_numpy()
    extent = max(float(np.nanmax(np.abs(values))), 1.0e-9)
    y = np.arange(len(data))
    style_publication_axes(ax, grid_axis='x')
    if np.nanmin(values) < 0:
        ax.barh(y, 2*extent, left=-extent, color=PAPER_TRACK_COLOR, edgecolor=PAPER_BAR_EDGE, linewidth=0.5, height=0.55)
        bars = ax.barh(y, values, color=color, edgecolor=PAPER_BAR_EDGE, height=0.55)
        ax.axvline(0, color=PAPER_PANEL_EDGE, linewidth=0.8)
        ax.set_xlim(-extent*1.1, extent*1.16)
    else:
        ax.barh(y, extent, color=PAPER_TRACK_COLOR, edgecolor=PAPER_BAR_EDGE, linewidth=0.5, height=0.55)
        bars = ax.barh(y, values, color=color, edgecolor=PAPER_BAR_EDGE, height=0.55)
        ax.set_xlim(0, extent*1.16)
    ax.set_yticks(y, [prettify_label(name, 24) for name in data['method']])
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.tick_params(axis='y', labelsize=9)
    for bar, value in zip(bars, values):
        x = value - .015*extent if value > .15*extent else value + .015*extent
        ax.text(x, bar.get_y()+bar.get_height()/2, fmt.format(value), ha='right' if value > .15*extent else 'left', va='center', fontsize=8)
fig.suptitle('Hardware-Aware RL Compared with Optimizers and Reward Oracle', fontsize=17, y=0.99, weight='semibold')
fig.tight_layout(rect=(0, 0, 1, 0.96))
save_figure(fig, 'optimizer_only_comparison_panel.pdf')
plt.show()
plt.close(fig)

if SAVE_OUTPUTS:
    comparison.to_csv(OUTPUT_DIR / f'{EVAL_SPLIT}_baseline_summary.csv', index=False)
    rl_selected.to_csv(OUTPUT_DIR / f'{EVAL_SPLIT}_rl_selections.csv', index=False)
print(f'Comparison plots and summary tables saved under {OUTPUT_DIR}')


## Per-Circuit Metric Differences vs the Best Training-Selected 2Q Chain

The fixed chain is chosen using training split results, then compared with the policy on the same held-out circuits. Positive differences favor the policy for reward, gate reduction, and depth reduction; negative runtime differences mean the policy is faster.


In [ ]:
comparison_baseline_chain = best_chain_for_two_qubit_reduction
comparison_baseline_label = best_twoq_chain_label
comparison_baseline_selected = selected_rows(
    eval_df, comparison_baseline_chain, comparison_baseline_label, 'Fixed chain'
)

rl_by_circuit = rl_selected.set_index('original_circuit')
baseline_by_circuit = comparison_baseline_selected.set_index('original_circuit')
circuit_metric_compare = pd.DataFrame(index=rl_by_circuit.index)
circuit_metric_compare['model_selected_chain'] = rl_by_circuit['selected_chain']
circuit_metric_compare['baseline_selected_chain'] = baseline_by_circuit['selected_chain']
circuit_metric_compare['utility_diff'] = rl_by_circuit['episode_reward'] - baseline_by_circuit['episode_reward']
circuit_metric_compare['twoq_reduction_diff'] = (
    rl_by_circuit['total_two_qubit_gate_reduction_pct'] - baseline_by_circuit['total_two_qubit_gate_reduction_pct']
)
circuit_metric_compare['depth_reduction_diff'] = (
    rl_by_circuit['total_depth_reduction_pct'] - baseline_by_circuit['total_depth_reduction_pct']
)
circuit_metric_compare['runtime_seconds_diff'] = (
    rl_by_circuit['episode_total_runtime_seconds'] - baseline_by_circuit['episode_total_runtime_seconds']
)
circuit_metric_compare['runtime_score_diff'] = (
    rl_by_circuit['episode_runtime_score'] - baseline_by_circuit['episode_runtime_score']
)
circuit_metric_compare = circuit_metric_compare.sort_values(
    ['utility_diff', 'twoq_reduction_diff', 'depth_reduction_diff'], ascending=True
)
comparison_display = circuit_metric_compare.reset_index().rename(columns={'index': 'original_circuit'})
if SAVE_OUTPUTS:
    comparison_display.to_csv(OUTPUT_DIR / 'per_circuit_model_vs_best_2q_chain.csv', index=False)
display(comparison_display)
print(
    f'Per-circuit comparison: RL policy vs {comparison_baseline_chain}; '
    f'{len(comparison_display)} circuits in the {EVAL_SPLIT} split.'
)

fig, axes = plt.subplots(2, 2, figsize=(max(14, min(24, .23 * len(circuit_metric_compare) + 8)), 9), facecolor=PAPER_FIGURE_COLOR)
metric_specs = [
    ('utility_diff', 'Utility difference', True),
    ('twoq_reduction_diff', '2Q reduction difference (percentage points)', True),
    ('depth_reduction_diff', 'Depth reduction difference (percentage points)', True),
    ('runtime_seconds_diff', 'Runtime difference (seconds; lower is better)', False),
]
for ax, (column, title, positive_is_good) in zip(axes.flat, metric_specs):
    style_publication_axes(ax, grid_axis='y')
    values = circuit_metric_compare[column].to_numpy(dtype=float)
    good_color = '#4F7F8C' if positive_is_good else '#B05A62'
    bad_color = '#B05A62' if positive_is_good else '#4F7F8C'
    ax.bar(np.arange(len(values)), values, color=np.where(values >= 0, good_color, bad_color), edgecolor=PAPER_BAR_EDGE, linewidth=.4, width=.82)
    ax.axhline(0, color=PAPER_PANEL_EDGE, linewidth=.9)
    ax.set_title(title, fontsize=12, pad=8, weight='semibold')
    ax.tick_params(axis='x', labelsize=7, rotation=90)
    ax.tick_params(axis='y', labelsize=8)
for ax in axes[2:]:
    ax.set_xlabel(f'{EVAL_SPLIT.title()} circuit', fontsize=10, labelpad=8)
fig.suptitle(
    f'Per-Circuit Differences: RL Policy vs {prettify_label(comparison_baseline_chain, None)}',
    fontsize=15, y=.995, weight='semibold'
)
fig.tight_layout(rect=(0, 0, 1, .96))
save_figure(fig, 'per_circuit_metric_difference_panel_best_2q_chain.pdf')
save_figure(fig, 'per_circuit_metric_difference_panel_best_2q_chain.png')
plt.show()
plt.close(fig)


## Test-set comparison: 2Q-focused RL policy and all fixed strategies

Circuit-level box plots compare the policy with the three single optimizers and nine ordered chains on the same 64 held-out test circuits. Boxes show the median and interquartile range; whiskers extend to 1.5 times the interquartile range, with outliers hidden for readability. Black diamonds and the values in the right column show the arithmetic mean. Higher two-qubit gate reduction is better. Color identifies the first optimizer in a chain.


In [ ]:
# Circuit-level distribution comparison for the held-out test split.
from matplotlib.lines import Line2D
if EVAL_SPLIT != 'test':
    raise ValueError('The all-strategy policy comparison must be run on the held-out test split.')
if (RUNTIME_WEIGHT, TWO_QUBIT_WEIGHT, DEPTH_WEIGHT, FINAL_REWARD_WEIGHT) != (0.0, 1.0, 0.0, 1.0):
    raise ValueError('The test figure requires the 2Q=1, depth=0, runtime=0, final=1 policy.')

strategy_order_raw = single_chains + double_chains
strategy_labels = {
    name: ' → '.join(OPTIMIZER_LABELS[part] for part in name.split('__'))
    for name in strategy_order_raw
}
fixed_test = eval_df.loc[
    eval_df['chain_name'].isin(strategy_order_raw),
    ['original_circuit', 'chain_name', 'total_two_qubit_gate_reduction_pct'],
].copy()
fixed_test['method'] = fixed_test['chain_name'].map(strategy_labels)
fixed_test['first_optimizer'] = fixed_test['chain_name'].map(
    lambda name: OPTIMIZER_LABELS[name.split('__')[0]]
)
policy_test = rl_selected[
    ['original_circuit', 'total_two_qubit_gate_reduction_pct']
].copy()
policy_test['method'] = 'RL policy'
policy_test['first_optimizer'] = 'RL policy'
policy_and_strategies = pd.concat([policy_test, fixed_test], ignore_index=True)
policy_strategy_order = ['RL policy'] + [strategy_labels[name] for name in strategy_order_raw]
expected_test_circuits = eval_df['original_circuit'].nunique()
if len(policy_test) != expected_test_circuits or expected_test_circuits != 64:
    raise ValueError(f'Expected one policy result per 64 test circuits; found {len(policy_test)} results for {expected_test_circuits} circuits.')
if policy_and_strategies.groupby('method')['original_circuit'].nunique().ne(64).any():
    raise ValueError('Every method must have an outcome for each of the 64 test circuits.')

wide_test = policy_and_strategies.pivot(
    index='original_circuit', columns='method', values='total_two_qubit_gate_reduction_pct'
).reindex(columns=policy_strategy_order)
if wide_test.isna().any().any():
    raise ValueError('The test comparison contains missing method outcomes.')
test_values = wide_test.to_numpy(dtype=float)

hardware_method_colors = {
    'Qiskit AI': '#356F9F',
    'Qiskit standard': '#3C8D75',
    'TKET': '#C87943',
    'RL policy': '#B55252',
}
fig, ax = plt.subplots(figsize=(7.9, 8.6), facecolor=PAPER_FIGURE_COLOR)
style_publication_axes(ax, grid_axis='x')
box_data = [wide_test[name].to_numpy(dtype=float) for name in policy_strategy_order]
boxplot = ax.boxplot(
    box_data, vert=False, patch_artist=True, showfliers=False, widths=0.62,
    medianprops={'color': '#26343D', 'linewidth': 1.15},
    whiskerprops={'color': '#657681', 'linewidth': 0.8},
    capprops={'color': '#657681', 'linewidth': 0.8},
    boxprops={'edgecolor': '#26343D', 'linewidth': 0.6},
)
for patch, method in zip(boxplot['boxes'], policy_strategy_order):
    if method == 'RL policy':
        color = hardware_method_colors['RL policy']
    else:
        color = hardware_method_colors[method.split(' → ')[0] if ' → ' in method else method]
    patch.set_facecolor(color)
    patch.set_alpha(0.62)
method_means = test_values.mean(axis=0)
method_positions = np.arange(1, len(policy_strategy_order) + 1)
ax.scatter(
    method_means, method_positions, marker='D', s=42,
    color='#17232B', edgecolor='white', linewidth=0.6, zorder=5,
)
for position, mean in zip(method_positions, method_means):
    ax.text(
        1.035, position, f'{mean:.1f}%',
        transform=ax.get_yaxis_transform(), ha='left', va='center',
        fontsize=20.0, color='black', clip_on=False,
    )
ax.text(
    1.035, 0.35, 'Mean', transform=ax.get_yaxis_transform(),
    ha='left', va='bottom', fontsize=20, weight='bold', clip_on=False,
)
ax.axvline(0, color=PAPER_PANEL_EDGE, linewidth=0.9, linestyle='--', zorder=1)
ax.set_yticks(np.arange(1, len(policy_strategy_order) + 1))
ax.set_yticklabels(policy_strategy_order, fontsize=14.0)
ax.invert_yaxis()
ax.set_xlabel('Two-qubit gate reduction from logical input (%)', fontsize=20)
ax.set_ylabel('')
ax.set_title('Two-qubit gate reduction', loc='center', fontsize=20, weight='bold')
fig.suptitle('Hardware-aware RL Performance', x=0.69, y=0.985, ha='center', fontsize=26.0, weight='bold', color=PAPER_LABEL_COLOR)
ax.tick_params(axis='x', labelsize=12.5)
ax.tick_params(axis='y', length=0, labelsize=20)
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
legend_handles = [
    Patch(facecolor=hardware_method_colors[name], edgecolor='#26343D', alpha=0.62, label=name)
    for name in ('RL policy', 'Qiskit AI', 'Qiskit standard', 'TKET')
]
mean_handle = Line2D([], [], marker='D', linestyle='None', color='#17232B', markeredgecolor='white', markersize=7, label='Mean')
fig.legend(handles=legend_handles + [mean_handle], title='First optimizer / policy', frameon=False,
           loc='lower center', bbox_to_anchor=(0.62, 0.012), fontsize=14.0,
           title_fontsize=15.0, ncol=5, columnspacing=1.0, handletextpad=0.4)
fig.subplots_adjust(left=0.41, right=0.82, top=0.88, bottom=0.17)
save_figure(fig, 'test_2q_policy_vs_12_strategies.pdf')
fig.savefig(OUTPUT_DIR / 'test_2q_policy_vs_12_strategies.png', bbox_inches='tight', dpi=300, facecolor=fig.get_facecolor())
policy_strategy_summary = pd.DataFrame({
    'method': policy_strategy_order,
    'mean_two_qubit_gate_reduction_pct': [float(np.mean(v)) for v in box_data],
    'median_two_qubit_gate_reduction_pct': [float(np.median(v)) for v in box_data],
    'q1_two_qubit_gate_reduction_pct': [float(np.percentile(v, 25)) for v in box_data],
    'q3_two_qubit_gate_reduction_pct': [float(np.percentile(v, 75)) for v in box_data],
    'test_circuits': len(test_values),
})
policy_strategy_summary.to_csv(OUTPUT_DIR / 'test_2q_policy_vs_12_strategies_summary.csv', index=False)
display(policy_strategy_summary)
plt.show()
plt.close(fig)
